# Guitar Teacher AI - Fine-tuning MAESTRO su Kaggle

Questo notebook permette di addestrare il modello MAESTRO sul dataset GAPS sfruttando le potentissime GPU gratuite fornite da Kaggle (es. P100 o T4x2).

**Istruzioni preliminari per Kaggle:**
1. Vai nel menu a destra su **Notebook options** (o Settings) > **Accelerator** > Seleziona **GPU T4 x2** oppure **GPU P100**.
2. Sempre a destra, fai clic su **Add Input** > **Upload** (icona a forma di nuvola/freccia) e crea un nuovo Dataset caricando il tuo file `maestro_model.zip` dal Mac. Dagli un nome qualsiasi, ad esempio `guitar-teacher-code`.

### 1. Copia del codice nell'ambiente di lavoro
Kaggle decomprime in automatico i file zip caricati e li mette nella cartella di sola lettura `/kaggle/input`. Noi dobbiamo copiare la cartella estratta `maestro_model` nella cartella di lavoro `/kaggle/working` dove lo script ha i permessi per scrivere (es. per salvare i file audio scaricati e i pesi del modello).

In [ ]:
import glob
import os
import shutil

# Cerca in automatico la cartella maestro_model dezippata da Kaggle negli input
# recursive=True cercherà anche se Kaggle l'ha messa in una sottocartella col nome del dataset
paths = glob.glob('/kaggle/input/**/maestro_model', recursive=True)

if not paths:
    print("\u26A0\uFE0F ATTENZIONE: Nessuna cartella 'maestro_model' trovata in /kaggle/input/.")
    print("Assicurati di aver caricato lo zip tramite il pulsante 'Add Input'.")
else:
    source_dir = paths[0]
    dest_dir = '/kaggle/working/maestro_model'
    
    print(f"Cartella trovata in: {source_dir}")
    print(f"Copia in {dest_dir} in corso...")
    
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)  # Svuota prima se esegui la cella due volte
        
    shutil.copytree(source_dir, dest_dir)
    print("Copia completata con successo! I permessi di scrittura sono abilitati.")

### 2. Installazione delle dipendenze
Kaggle ha già quasi tutto pre-installato, ma ci assicuriamo di avere le librerie musicali e Hugging Face.

In [ ]:
!pip install librosa soundfile pretty_midi mir_eval numpy datasets huggingface_hub torchlibrosa

### 3. Avvio del Training
Adesso eseguiamo lo script. Al primo avvio lo script capirà che la cartella GAPS è vuota e scaricherà in automatico i file audio wav usando Hugging Face a grandissima velocità. Poi inizierà ad addestrare le epoche!

In [ ]:
import os

# Spostiamoci nella cartella di lavoro principale
os.chdir('/kaggle/working')

# Assicuriamoci che python riesca a importare maestro_model
import sys
if '/kaggle/working' not in sys.path:
    sys.path.append('/kaggle/working')

# Lanciamo lo script di fine-tuning!
!python -m maestro_model.train_finetune

### 4. Download dei pesi finali
Su Kaggle, tutto ciò che viene salvato in `/kaggle/working` compare nel pannello laterale a destra **"Output"**.

Quando il training finisce (o se decidi di interromperlo dopo un po' di epoche per usare i pesi intermedi):
1. Vai nel pannello di destra sotto la voce **Output** > `/kaggle/working/maestro_model/weights/finetuned/`.
2. Trova il file `best_model.pth`.
3. Fai clic sui **tre pallini** accanto al file e seleziona **Download** per salvarlo sul tuo Mac!